In [ ]:
# hide
import numpy as np
import pyquist as pq


def iter_frames(audio, hop_length, frame_length):
    x = np.asarray(audio.samples).reshape(-1)
    for start in range(0, len(x), hop_length):
        yield x[start:start + frame_length]


def overlap_add(frames, hop_length, sample_rate):
    frames = [f for f in frames]
    frame_length = len(frames[0])
    out = np.zeros(hop_length * (len(frames) - 1) + frame_length)
    for k, frame in enumerate(frames):
        if len(frame) == frame_length:
            out[k * hop_length:k * hop_length + frame_length] += frame
    return pq.Audio(out.astype(np.float32), sample_rate)


def hann(n):
    return 0.5 * (1 - np.cos(2 * np.pi * np.arange(n) / n))

In [ ]:
# Granular synthesis: chop the sound into overlapping grains, MANIPULATE them,
# and glue them back. Edit `manipulate` to invent your own effect!
def manipulate(grains):
    grains = [g * hann(len(g)) for g in grains]     # smooth each grain's edges
    out = []                                        # shuffle order within blocks
    for i in range(0, len(grains), 100):
        block = grains[i:i + 100]
        np.random.shuffle(block)
        out += block
    return out


grain_length = 2048            # grain size in samples (~46 ms)
hop_length = 1024              # spacing when extracting grains
overlap_hop = 1024            # spacing when reassembling (change to time-stretch!)

audio = pq.Audio.from_file("../assets/audio-trio.wav")
grains = [g for g in iter_frames(audio, hop_length, grain_length) if len(g) == grain_length]
grains = manipulate(grains)
pq.play(overlap_add(grains, overlap_hop, audio.sample_rate))